# Práctica · Semana 7, parte A: Redes neuronales en Python

**Curso:** Introducción a la Inteligencia Artificial · UDES
**Tiempo estimado:** 1 h 45 min
**Relación con el capítulo:** secciones 7.1 a 7.5

Esta es la primera parte de la práctica de la semana. Vas a construir **desde cero**, solo con `numpy`, las redes neuronales que aparecen en el capítulo, para entender qué ocurre dentro de ellas antes de usarlas como cajas negras en TensorFlow Playground y Teachable Machine (partes B y C, en el [README de la práctica](README.md)).

| Sección | Qué construyes | Idea clave |
|---|---|---|
| 1 | Una neurona de McCulloch y Pitts | Una neurona calcula una suma ponderada y la compara con un umbral |
| 2 | Un perceptrón que aprende AND, OR y XOR | La regla del perceptrón y el límite de la separabilidad lineal |
| 3 | Un perceptrón multicapa con retropropagación | Las capas ocultas resuelven XOR |
| 4 | Una red de Hopfield | Memoria asociativa: recordar un patrón a partir de una versión dañada |
| 5 | Un clasificador de dígitos escritos a mano | Una red más grande con scikit-learn |

Ejecuta las celdas en orden. Las celdas marcadas con ✏️ son ejercicios. Todas las celdas se probaron con Python 3 y usan librerías que Google Colab trae instaladas.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

np.set_printoptions(precision=3, suppress=True)

## 1. La neurona artificial

McCulloch y Pitts (1943) propusieron el primer modelo de neurona artificial: recibe entradas $x_i$, las multiplica por unos **pesos** $w_i$, suma el resultado con un **sesgo** $b$ y se «activa» (devuelve 1) si la suma supera 0 (sección 7.1).

$$
y = \begin{cases} 1 & \text{si } \sum_i w_i x_i + b > 0 \\ 0 & \text{en otro caso} \end{cases}
$$

Con los pesos adecuados, una sola neurona calcula funciones lógicas.

In [ ]:
def neurona(x, w, b):
    return int(np.dot(w, x) + b > 0)


ENTRADAS = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
for nombre, w, b in [("AND", [1, 1], -1.5), ("OR", [1, 1], -0.5), ("NOT x1", [-1, 0], 0.5)]:
    print(nombre.ljust(7), [neurona(x, w, b) for x in ENTRADAS])

### ✏️ Ejercicio 1. Pesos a mano

1. Encuentra pesos y sesgo para la función **NAND** (el negado de AND) y compruébalos.
2. Intenta encontrar pesos para **XOR** (1 solo cuando las entradas son distintas). ¿Lo logras? Dibuja los cuatro puntos en un plano y explica por qué ninguna recta separa los 1 de los 0.

In [ ]:
# Escribe aquí tus pesos para NAND

*Tus respuestas al ejercicio 1:*

## 2. El perceptrón: una neurona que aprende

Rosenblatt (1958) añadió a la neurona una **regla de aprendizaje**. Para cada ejemplo, si la salida $y$ es distinta de la deseada $t$, corrige los pesos en la dirección del error (sección 7.2):

$$
w_i \leftarrow w_i + \eta\,(t - y)\,x_i \qquad b \leftarrow b + \eta\,(t - y)
$$

donde $\eta$ es la **tasa de aprendizaje**. Repetimos el recorrido de los datos (cada pasada es una **época**) hasta que no haya errores o se acabe el límite de épocas.

In [ ]:
def entrenar_perceptron(X, t, eta=0.1, epocas=20, semilla=0):
    rng = np.random.default_rng(semilla)
    w = rng.normal(0, 0.5, X.shape[1])
    b = 0.0
    errores_por_epoca = []
    for _ in range(epocas):
        errores = 0
        for x, objetivo in zip(X, t):
            y = int(np.dot(w, x) + b > 0)
            if y != objetivo:
                w += eta * (objetivo - y) * x
                b += eta * (objetivo - y)
                errores += 1
        errores_por_epoca.append(errores)
        if errores == 0:
            break
    return w, b, errores_por_epoca


def frontera(ax, w, b, titulo):
    ax.set_xlim(-0.5, 1.5)
    ax.set_ylim(-0.5, 1.5)
    xs = np.linspace(-0.5, 1.5, 10)
    if abs(w[1]) > 1e-9:
        ax.plot(xs, -(w[0] * xs + b) / w[1], "k--")
    ax.set_title(titulo)


FUNCIONES = {"AND": [0, 0, 0, 1], "OR": [0, 1, 1, 1], "XOR": [0, 1, 1, 0]}
fig, ejes = plt.subplots(1, 3, figsize=(12, 3.6))
for ax, (nombre, salida) in zip(ejes, FUNCIONES.items()):
    t = np.array(salida)
    w, b, errores = entrenar_perceptron(ENTRADAS, t, epocas=50)
    pred = [int(np.dot(w, x) + b > 0) for x in ENTRADAS]
    ax.scatter(*ENTRADAS.T, c=t, cmap="bwr", s=150, edgecolors="k")
    frontera(ax, w, b, f"{nombre}: {sum(p == o for p, o in zip(pred, t))}/4 aciertos")
    print(f"{nombre}: épocas = {len(errores)}, errores en la última época = {errores[-1]}, "
          f"w = {w}, b = {b:.2f}")
plt.tight_layout()
plt.show()

El perceptrón aprende AND y OR en pocas épocas, pero **nunca** aprende XOR: los errores no bajan a cero aunque le demos 50 épocas. El **teorema de convergencia del perceptrón** garantiza que encuentra una solución solo si los datos son **linealmente separables** (sección 7.2.3). Minsky y Papert (1969) usaron este límite para argumentar contra las redes de una capa, y el interés por las redes neuronales cayó durante más de una década.

### ✏️ Ejercicio 2. La tasa de aprendizaje

1. Entrena el perceptrón para OR con tasas de aprendizaje de 0,01, 0,1 y 1. ¿Cambia el número de épocas necesarias? ¿Cambia la recta final?
2. ¿Por qué cambian los resultados si cambias la `semilla`?

In [ ]:
for eta in [0.01, 0.1, 1.0]:
    w, b, errores = entrenar_perceptron(ENTRADAS, np.array(FUNCIONES["OR"]), eta=eta, epocas=100)
    print(f"eta = {eta:<5} épocas = {len(errores):>3}  errores por época = {errores}")

*Tus respuestas al ejercicio 2:*

## 3. Un perceptrón multicapa con retropropagación

Para resolver XOR hace falta una **capa oculta**. Usamos una red 2-4-1: dos entradas, cuatro neuronas ocultas y una salida. Cambiamos el escalón por la función **sigmoide** $\sigma(z) = 1/(1 + e^{-z})$, que es derivable, y entrenamos con **retropropagación** (Rumelhart et al., 1986; sección 7.3):

1. **Propagación hacia adelante:** calcular la salida de la red.
2. **Error:** comparar la salida con el valor deseado.
3. **Propagación hacia atrás:** calcular, con la regla de la cadena, cuánto contribuyó cada peso al error (el **gradiente**).
4. **Descenso del gradiente:** mover cada peso un poco en la dirección que reduce el error.

In [ ]:
def sigmoide(z):
    return 1 / (1 + np.exp(-z))


class RedMulticapa:
    # Red de una capa oculta con activación sigmoide, entrenada con retropropagación.

    def __init__(self, n_entradas, n_ocultas, n_salidas=1, semilla=0):
        rng = np.random.default_rng(semilla)
        self.W1 = rng.normal(0, 1, (n_entradas, n_ocultas))
        self.b1 = np.zeros(n_ocultas)
        self.W2 = rng.normal(0, 1, (n_ocultas, n_salidas))
        self.b2 = np.zeros(n_salidas)

    def adelante(self, X):
        self.h = sigmoide(X @ self.W1 + self.b1)          # capa oculta
        self.y = sigmoide(self.h @ self.W2 + self.b2)     # capa de salida
        return self.y

    def entrenar(self, X, T, eta=0.5, epocas=5000):
        perdidas = []
        T = T.reshape(-1, 1)
        for _ in range(epocas):
            Y = self.adelante(X)
            perdidas.append(float(np.mean((Y - T) ** 2)))  # error cuadrático medio
            # Propagación hacia atrás (regla de la cadena)
            d_salida = (Y - T) * Y * (1 - Y)
            d_oculta = (d_salida @ self.W2.T) * self.h * (1 - self.h)
            # Descenso del gradiente
            self.W2 -= eta * self.h.T @ d_salida / len(X)
            self.b2 -= eta * d_salida.mean(axis=0)
            self.W1 -= eta * X.T @ d_oculta / len(X)
            self.b1 -= eta * d_oculta.mean(axis=0)
        return perdidas


T_xor = np.array(FUNCIONES["XOR"])
red = RedMulticapa(2, 4, semilla=1)
perdidas = red.entrenar(ENTRADAS, T_xor, eta=2.0, epocas=5000)
print("Salidas de la red para XOR:", red.adelante(ENTRADAS).ravel().round(3))
print("Redondeadas:               ", (red.adelante(ENTRADAS).ravel() > 0.5).astype(int), "  deseadas:", T_xor)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6))
a1.plot(perdidas)
a1.set_xlabel("Época")
a1.set_ylabel("Error cuadrático medio")
a1.set_title("Curva de aprendizaje")
g = np.linspace(-0.5, 1.5, 200)
GX, GY = np.meshgrid(g, g)
Z = red.adelante(np.c_[GX.ravel(), GY.ravel()]).reshape(GX.shape)
a2.contourf(GX, GY, Z, levels=20, cmap="bwr", alpha=0.6)
a2.scatter(*ENTRADAS.T, c=T_xor, cmap="bwr", s=150, edgecolors="k")
a2.set_title("Frontera de decisión aprendida para XOR")
plt.tight_layout()
plt.show()

La red multicapa aprende XOR: la capa oculta transforma el problema en uno que la neurona de salida sí puede separar. La frontera ya no es una recta.

### 3.1 Un problema más difícil: dos lunas entrelazadas

Probemos con un conjunto de datos de dos «lunas» entrelazadas, como los de TensorFlow Playground. Separamos el 30 % de los puntos como prueba y comparamos redes con distinto número de neuronas ocultas.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

X, t = make_moons(n_samples=400, noise=0.2, random_state=0)
X = (X - X.mean(axis=0)) / X.std(axis=0)                  # normalizar ayuda a aprender
X_tr, X_te, t_tr, t_te = train_test_split(X, t, test_size=0.3, random_state=0)

fig, ejes = plt.subplots(1, 4, figsize=(15, 3.4))
g = np.linspace(-2.5, 2.5, 150)
GX, GY = np.meshgrid(g, g)
for ax, n_ocultas in zip(ejes, [1, 2, 4, 16]):
    red = RedMulticapa(2, n_ocultas, semilla=0)
    red.entrenar(X_tr, t_tr, eta=3.0, epocas=8000)
    exac_tr = np.mean((red.adelante(X_tr).ravel() > 0.5) == t_tr)
    exac_te = np.mean((red.adelante(X_te).ravel() > 0.5) == t_te)
    Z = red.adelante(np.c_[GX.ravel(), GY.ravel()]).reshape(GX.shape)
    ax.contourf(GX, GY, Z, levels=20, cmap="bwr", alpha=0.5)
    ax.scatter(*X_te.T, c=t_te, cmap="bwr", s=12, edgecolors="k", linewidths=0.3)
    ax.set_title(f"{n_ocultas} ocultas: entren. {exac_tr:.2f} / prueba {exac_te:.2f}", fontsize=9)
    print(f"{n_ocultas:>2} neuronas ocultas: exactitud entrenamiento {exac_tr:.3f} | prueba {exac_te:.3f}")
plt.tight_layout()
plt.show()

### ✏️ Ejercicio 3. Capacidad y sobreajuste

1. ¿Cuántas neuronas ocultas necesita la red para separar bien las lunas? ¿Qué forma tiene la frontera con 1 neurona? ¿Por qué?
2. Cambia `noise=0.2` por `noise=0.4` y vuelve a ejecutar. ¿Aumenta la diferencia entre la exactitud de entrenamiento y la de prueba con 16 neuronas? Relaciona tu respuesta con el **sobreajuste** (sección 6.2.4).
3. Prueba distintas tasas de aprendizaje (`eta`) y números de épocas. ¿Qué pasa si `eta` es muy grande?

*Tus respuestas al ejercicio 3:*

## 4. La red de Hopfield: una memoria asociativa

Una **red de Hopfield** (Hopfield, 1982) no clasifica: **recuerda**. Todas sus neuronas están conectadas entre sí, y sus pesos se fijan con la **regla de Hebb** para que los patrones almacenados sean estados estables. Si se le presenta una versión **dañada** de un patrón, la red evoluciona hasta el patrón completo más parecido (sección 7.4).

Cada patrón es una imagen de 5 × 5 píxeles con valores +1 (negro) y −1 (blanco).

In [ ]:
def a_patron(dibujo):
    return np.array([1 if c == "#" else -1 for fila in dibujo for c in fila])


LETRAS = {
    "T": ["#####", "..#..", "..#..", "..#..", "..#.."],
    "L": ["#....", "#....", "#....", "#....", "#####"],
    "X": ["#...#", ".#.#.", "..#..", ".#.#.", "#...#"],
}
patrones = {k: a_patron(v) for k, v in LETRAS.items()}


class Hopfield:
    def __init__(self, n):
        self.W = np.zeros((n, n))

    def almacenar(self, lista):
        for p in lista:                           # regla de Hebb: refuerza neuronas que coinciden
            self.W += np.outer(p, p)
        np.fill_diagonal(self.W, 0)
        self.W /= len(lista)

    def energia(self, s):
        return -0.5 * s @ self.W @ s

    def recordar(self, s, pasos=10, semilla=0):
        rng = np.random.default_rng(semilla)
        s = s.copy()
        energias = [self.energia(s)]
        for _ in range(pasos):
            for i in rng.permutation(len(s)):    # actualización asíncrona
                s[i] = 1 if self.W[i] @ s >= 0 else -1
            energias.append(self.energia(s))
        return s, energias


def ensuciar(p, n_cambios, semilla):
    rng = np.random.default_rng(semilla)
    q = p.copy()
    idx = rng.choice(len(p), n_cambios, replace=False)
    q[idx] *= -1
    return q


red_h = Hopfield(25)
red_h.almacenar(list(patrones.values()))

fig, ejes = plt.subplots(3, 3, figsize=(6, 6))
for fila, (letra, p) in enumerate(patrones.items()):
    sucio = ensuciar(p, 6, semilla=fila)
    recordado, energias = red_h.recordar(sucio)
    for col, (img, titulo) in enumerate([(p, f"original {letra}"), (sucio, "con 6 píxeles cambiados"),
                                          (recordado, "recordado")]):
        ejes[fila, col].imshow(img.reshape(5, 5), cmap="gray_r")
        ejes[fila, col].set_title(titulo, fontsize=8)
        ejes[fila, col].axis("off")
    print(f"{letra}: ¿recuperó el original? {np.array_equal(recordado, p)} | energía {energias[0]:.1f} -> {energias[-1]:.1f}")
plt.tight_layout()
plt.show()

### ✏️ Ejercicio 4. ¿Cuánto puede recordar?

1. Aumenta el número de píxeles cambiados (6, 8, 10, 12). ¿A partir de cuántos la red deja de recuperar las letras? ¿Qué recuerda entonces?
2. Ejecuta la celda siguiente, que almacena cada vez más patrones aleatorios en una red de 100 neuronas y mide cuántos se mantienen estables. Hopfield (1982) estimó que una red de $N$ neuronas almacena de forma fiable unos $0{,}15\,N$ patrones. ¿Se cumple?
3. ¿Por qué la energía nunca aumenta durante el recuerdo? ¿Qué relación tiene esto con el *hill climbing* de la semana 3?

In [ ]:
rng = np.random.default_rng(0)
N = 100
for k in [5, 10, 14, 20, 30]:
    lista = [rng.choice([-1, 1], N) for _ in range(k)]
    h = Hopfield(N)
    h.almacenar(lista)
    estables = sum(np.array_equal(h.recordar(p, pasos=5)[0], p) for p in lista)
    print(f"{k:>2} patrones almacenados: {estables:>2} se mantienen estables ({100 * estables / k:.0f} %)")

*Tus respuestas al ejercicio 4:*

## 5. Hacia el aprendizaje profundo: reconocer dígitos escritos a mano

Las redes de las secciones anteriores tenían unas pocas neuronas. Para reconocer imágenes hacen falta muchas más. Usamos el conjunto de datos de **dígitos** de scikit-learn: 1.797 imágenes de 8 × 8 píxeles de dígitos escritos a mano, y una red multicapa ya implementada (`MLPClassifier`), que usa la misma retropropagación de la sección 3 con optimizaciones (sección 7.5).

In [ ]:
from sklearn.datasets import load_digits
from sklearn.metrics import confusion_matrix
from sklearn.neural_network import MLPClassifier

digitos = load_digits()
X_tr, X_te, y_tr, y_te = train_test_split(digitos.data / 16, digitos.target, test_size=0.3, random_state=0)

fig, ejes = plt.subplots(1, 8, figsize=(10, 1.6))
for ax, img, etiqueta in zip(ejes, digitos.images, digitos.target):
    ax.imshow(img, cmap="gray_r")
    ax.set_title(etiqueta)
    ax.axis("off")
plt.show()

mlp = MLPClassifier(hidden_layer_sizes=(64,), max_iter=500, random_state=0).fit(X_tr, y_tr)
print(f"Exactitud en prueba: {mlp.score(X_te, y_te):.3f}  ({len(y_te)} imágenes que la red no vio)")
print("Parámetros aprendidos:", sum(w.size for w in mlp.coefs_) + sum(b.size for b in mlp.intercepts_))
print("\nMatriz de confusión (filas = real, columnas = predicho):")
print(confusion_matrix(y_te, mlp.predict(X_te)))

In [ ]:
pred = mlp.predict(X_te)
errores = np.where(pred != y_te)[0][:8]
fig, ejes = plt.subplots(1, len(errores), figsize=(10, 1.8))
for ax, i in zip(np.atleast_1d(ejes), errores):
    ax.imshow(X_te[i].reshape(8, 8), cmap="gray_r")
    ax.set_title(f"real {y_te[i]}\npred. {pred[i]}", fontsize=8)
    ax.axis("off")
plt.show()

### ✏️ Ejercicio 5. Errores de la red

1. Observa las imágenes mal clasificadas. ¿Una persona las habría reconocido? ¿Qué dígitos confunde más la red según la matriz de confusión?
2. Cambia `hidden_layer_sizes` a `(8,)`, `(64,)` y `(128, 64)` (dos capas ocultas). ¿Cómo cambian la exactitud y el número de parámetros?
3. La red no «sabe» que la imagen es una imagen: recibe 64 números sueltos. ¿Qué información pierde? Relaciona tu respuesta con las **redes convolucionales** de la sección 7.7.

*Tus respuestas al ejercicio 5:*

## 6. Cierre

Completa esta tabla y continúa con las partes B (TensorFlow Playground) y C (Teachable Machine) del [README de la práctica](README.md).

| Red | ¿Qué hace? | ¿Cómo aprende o se configura? | Una limitación |
|---|---|---|---|
| Neurona de McCulloch y Pitts | | | |
| Perceptrón | | | |
| Perceptrón multicapa | | | |
| Red de Hopfield | | | |

---

**Referencias**

Hopfield, J. J. (1982). Neural networks and physical systems with emergent collective computational abilities. *Proceedings of the National Academy of Sciences, 79*(8), 2554-2558. https://doi.org/10.1073/pnas.79.8.2554

McCulloch, W. S., y Pitts, W. (1943). A logical calculus of the ideas immanent in nervous activity. *The Bulletin of Mathematical Biophysics, 5*, 115-133. https://doi.org/10.1007/BF02478259

Minsky, M., y Papert, S. (1969). *Perceptrons: An introduction to computational geometry*. MIT Press.

Rosenblatt, F. (1958). The perceptron: A probabilistic model for information storage and organization in the brain. *Psychological Review, 65*(6), 386-408. https://doi.org/10.1037/h0042519

Rumelhart, D. E., Hinton, G. E., y Williams, R. J. (1986). Learning representations by back-propagating errors. *Nature, 323*, 533-536. https://doi.org/10.1038/323533a0